# 06 — Meta Prophet Forecasting & Event Modeling
## Corporación Favorita Grocery Sales Forecasting

### Purpose of this Notebook
This notebook builds and evaluates **Meta Prophet** models:
1. **Decomposable Time-Series Modeling:** $y(t) = g(t) + s(t) + h(t) + \epsilon_t$ (Trend, Seasonality, Holidays).
2. **Prophet Basic (Additive Mode):** Baseline modeling weekly and annual seasonality without explicit event mapping.
3. **Prophet + Holidays (Multiplicative Mode):** Incorporating cleaned national and regional holiday schedules with custom event windows ($[-2, +1]$) and multiplicative scaling.
4. **Algorithmic Changepoint Analysis:** Detecting macroeconomic breaks (the 2014 oil crash and the April 2016 earthquake).
5. **Component Visualization:** Generating and saving component decomposition plots.
6. **Artifact Export:** Serializing `reports/prophet_summary.json`, `reports/prophet_forecast.csv`, and `reports/prophet_components_holidays.png`.


In [ ]:
import os
import sys
import json
import logging
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')
logging.getLogger('prophet').setLevel(logging.WARNING)

plt.style.use('seaborn-v0_8-whitegrid')

# --- 1. Universal Path Resolution ---
if os.path.exists('/content/drive/MyDrive/Sales-Forecasting_Project'):
    BASE_DIR = Path('/content/drive/MyDrive/Sales-Forecasting_Project/sales-forecasting')
else:
    cwd = Path.cwd().resolve()
    BASE_DIR = cwd.parent if cwd.name == 'notebooks' else cwd

DATA_DIR = BASE_DIR / 'data'
REPORTS_DIR = BASE_DIR / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loading cleaned dataset from {DATA_DIR / 'clean_merged.csv'}...")
df = pd.read_csv(DATA_DIR / 'clean_merged.csv', parse_dates=['date'])
print(f"Loaded {df.shape[0]:,} rows.")


### 2. Data Preparation for Prophet
Prophet requires column names `ds` (date) and `y` (target metric).


In [ ]:
daily_sales = df.groupby('date')['sales'].sum().reset_index()
daily_sales = daily_sales.rename(columns={'date': 'ds', 'sales': 'y'})

split_idx = int(len(daily_sales) * 0.8)
prophet_train = daily_sales.iloc[:split_idx].copy()
prophet_test = daily_sales.iloc[split_idx:].copy()

print(f"Prophet train rows: {len(prophet_train)} | test rows: {len(prophet_test)}")


### 3. Model 1: Prophet Basic (Additive Seasonality)
Baseline Prophet with automatic yearly and weekly seasonalities.


In [ ]:
print("Training Basic Prophet (Additive)...")
m_basic = Prophet(yearly_seasonality=True, weekly_seasonality=True, seasonality_mode='additive')
m_basic.fit(prophet_train)

forecast_basic = m_basic.predict(prophet_test[['ds']])

mae_basic = mean_absolute_error(prophet_test['y'], forecast_basic['yhat'])
rmse_basic = np.sqrt(mean_squared_error(prophet_test['y'], forecast_basic['yhat']))
mape_basic = np.mean(np.abs((prophet_test['y'] - forecast_basic['yhat']) / prophet_test['y'])) * 100

print(f"Basic Prophet Metrics: MAE = {mae_basic:,.2f} | RMSE = {rmse_basic:,.2f} | MAPE = {mape_basic:.2f}%")


### 4. Model 2: Prophet + Cleaned Holidays (Multiplicative Seasonality)
We prepare the holiday schedule from `holidays_events.csv`, filter non-transferred national and regional holidays, set an anticipation window $[-2, +1]$, and switch to multiplicative seasonality.


In [ ]:
holidays_raw = pd.read_csv(DATA_DIR / 'holidays_events.csv', parse_dates=['date'])
valid_holidays = holidays_raw[holidays_raw['transferred'] == False].copy()
valid_holidays = valid_holidays[valid_holidays['locale'].isin(['National', 'Regional'])]

prophet_holidays = valid_holidays[['date', 'type']].rename(columns={'date': 'ds', 'holiday': 'holiday_name'})
prophet_holidays['holiday'] = prophet_holidays['type']
prophet_holidays['lower_window'] = -2
prophet_holidays['upper_window'] = 1
prophet_holidays = prophet_holidays[['ds', 'holiday', 'lower_window', 'upper_window']].drop_duplicates(subset=['ds'])

print(f"Configured {len(prophet_holidays)} holiday dates for Prophet.")

print("Training Prophet with Holidays & Multiplicative Seasonality...")
m_holidays = Prophet(
    holidays=prophet_holidays,
    yearly_seasonality=True,
    weekly_seasonality=True,
    seasonality_mode='multiplicative'
)
m_holidays.fit(prophet_train)

forecast_hol = m_holidays.predict(prophet_test[['ds']])

mae_hol = mean_absolute_error(prophet_test['y'], forecast_hol['yhat'])
rmse_hol = np.sqrt(mean_squared_error(prophet_test['y'], forecast_hol['yhat']))
mape_hol = np.mean(np.abs((prophet_test['y'] - forecast_hol['yhat']) / prophet_test['y'])) * 100

print(f"Prophet + Holidays Metrics: MAE = {mae_hol:,.2f} | RMSE = {rmse_hol:,.2f} | MAPE = {mape_hol:.2f}%")
print(f"Improvement over Basic:     {((mae_basic - mae_hol) / mae_basic) * 100:.2f}% MAE reduction!")


### 5. Algorithmic Changepoint Detection
Prophet automatically places potential changepoints throughout the time series.
We inspect the structural shifts identified near the late-2014 oil crash and April 2016 earthquake.


In [ ]:
print(f"Total changepoints evaluated: {len(m_holidays.changepoints)}")
print("\nRecent structural changepoints detected by Prophet:")
print(m_holidays.changepoints.tail(10))


### 6. Component Plotting & Export
We plot the decomposed components (Trend, Holidays, Weekly, Yearly) and export the visualization to `reports/prophet_components_holidays.png`.


In [ ]:
future_full = m_holidays.make_future_dataframe(periods=len(prophet_test), freq='D')
forecast_full = m_holidays.predict(future_full)

fig = m_holidays.plot_components(forecast_full)
fig_path = REPORTS_DIR / 'prophet_components_holidays.png'
fig.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved component decomposition plot to: {fig_path}")


### 7. Exporting Prophet Summary & Forecast Artifacts
Export `reports/prophet_summary.json` (metadata for Tab 4) and `reports/prophet_forecast.csv`.


In [ ]:
# 1. Summary JSON
prophet_summary = {
    'growth': m_holidays.growth,
    'seasonalities': list(m_holidays.seasonalities.keys()),
    'n_holidays': len(m_holidays.holidays) if m_holidays.holidays is not None else 0,
    'n_changepoints': len(m_holidays.changepoints),
    'mae': float(mae_hol),
    'rmse': float(rmse_hol),
    'mape': float(mape_hol)
}

summary_json_path = REPORTS_DIR / 'prophet_summary.json'
with open(summary_json_path, 'w', encoding='utf-8') as f:
    json.dump(prophet_summary, f, indent=2)
print(f"Saved summary JSON: {summary_json_path}")

# 2. Forecast CSV
prophet_forecast_df = pd.DataFrame({
    'date': prophet_test['ds'],
    'actual': prophet_test['y'].values,
    'predicted': forecast_hol['yhat'].values,
    'lower_ci': forecast_hol['yhat_lower'].values,
    'upper_ci': forecast_hol['yhat_upper'].values
})
forecast_csv_path = REPORTS_DIR / 'prophet_forecast.csv'
prophet_forecast_df.to_csv(forecast_csv_path, index=False)
print(f"Saved forecast CSV: {forecast_csv_path}")
print("\nProphet pipeline completed successfully! Proceed to 07_lightgbm_forecasting.ipynb.")
